# The data contract and the panel's own table, as queries

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.data.sql`

**Modules covered** `data/sql.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The panel is read by pandas, and the same work is stated as SQL over the snapshot's own files: the as-of join, the coverage report, and the assembly of the monthly euro excess return table the factor model is fitted on. The entry point runs all three and prints whether each agrees with the pandas path, which they are asserted to do on the frozen snapshot.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `data/sql.py::agreement` traces to the check that the two statements of the contract agree, which is what makes the query a statement of the interface rather than a second implementation of the panel
- `data/sql.py::as_of` traces to the as-of gate as one predicate, `available_from <= when`: a bar labelled with a month is readable only from the first day of the following month, which is the difference between a monthly label and a monthly bar
- `data/sql.py::connection` traces to the table contract of this effort stated a second time as SQL: the same as-of rule and coverage report the pandas path carries, asserted against it on the frozen snapshot rather than adopted by it, because the contract another project satisfies is a table contract and a query states it in a form that can be read without reading Python
- `data/sql.py::coverage` traces to the coverage aggregation of this effort in SQL, with the months a moment can actually see beside the months carried
- `data/sql.py::main` traces to the table contract of this effort stated a second time as SQL: the same as-of rule and coverage report the pandas path carries, asserted against it on the frozen snapshot rather than adopted by it, because the contract another project satisfies is a table contract and a query states it in a form that can be read without reading Python
- `data/sql.py::returns` traces to the assembly of the monthly euro excess return table of this effort, stated as one query: the price and currency legs as month-on-month ratios, the currency translation as `(1 + r) / (1 + fx) - 1` rather than a sum, the overnight rate compounded within the month at /360 on the month's own calendar days, and every leg under the same availability gate
- `data/sql.py::returns_agreement` traces to the check that the query's table and the pandas path's agree cell by cell, which is what lets the statement be the contract's definition rather than a paraphrase

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`data/sql.py`**

The table contract, expressed as a query over the snapshot's own files.

The panel is read by pandas, and this module states the same operations the analytics depend on (the
as-of join, the coverage report, and the assembly of the monthly euro excess return table) as SQL,
then checks the statements against each other on the frozen snapshot. The point is not that a query is
faster than a dataframe. It is that the contract another project has to satisfy is a *table* contract,
and a contract written only as pandas code is one the sibling has to re-derive rather than satisfy: a
query names the columns, the date arithmetic, the three-source join and the availability rule in a form
that can be read without reading Python.

**The assembly is the part worth stating this way.** A reader told the package works on a monthly euro
excess return table has, in the pandas path, to follow the currency translation, the return of holding
that currency, the accrual of an annualised overnight rate over a month's own calendar days and the
availability gate through four modules to find out what the table is. Here it is one statement: the
price legs and the currency legs reduced to month-on-month ratios, the cash leg compounded within the
month at /360 on the calendar days the publisher's unit implies, the translation as
`(1 + r) / (1 + fx) - 1` rather than a sum, and every leg gated by the same rule.

**The availability rule is stated in SQL here, not imported from pandas.** `period_month` is the
month a bar describes and `available_from` is the first instant it may be used, and expressing that
arithmetic a second time is the point of the check: if the two statements ever disagree, the check
fails, and a disagreement between them is exactly the look-ahead class of error the rule exists to
prevent. What is *not* duplicated is the rule's definition as a number: the pandas path stays the
authority the analytics run on, and this module's queries are asserted against it rather than adopted
by it.

Nothing in the analytics imports this module. It is a second statement of the data contract for a
reader and for the sibling project, not a second path the results travel down.

## 3. The data contract it consumes, and the as-of rule

The same contract, in the query's arithmetic: `period_month` is `date_trunc('month', date)` and `available_from` is that month plus one, so a bar labelled with a month becomes readable on the first day of the next. The priced legs are read as one long table by union, the gate is `available_from <= the moment the reader stands at`, and the coverage query reports the months carried beside the months visible at that moment. The pandas path stays the authority the analytics run on; the query is checked against it. The third statement is the table itself rather than its columns: the price and currency legs reduced to month-on-month ratios, the currency translation as `(1 + r) / (1 + fx) - 1` rather than a sum, the overnight rate compounded within the month at /360 on the month's own calendar days, and every leg under the same gate and window.

## 4. The worked example on small numbers, with the identity checked

The rule's arithmetic is the thing worth checking twice, so the worked example states it in both languages on the same three dated bars and asserts that the two agree.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import duckdb
import pandas as pd

from portfolio_workbench.data import panel

bars = pd.DataFrame({"date": pd.to_datetime(["2020-01-15", "2020-02-15", "2020-03-15"])})
pandas_rule = panel.available_from(pd.PeriodIndex(bars["date"], freq="M"))

connection = duckdb.connect()
connection.register("bar", bars)
queried = connection.execute(
    "SELECT date_trunc('month', bar.date) + INTERVAL 1 MONTH AS available_from FROM bar"
).fetch_df()

assert list(pd.to_datetime(queried["available_from"])) == list(pandas_rule)
print("both statements put a month's bar in the hands of the reader from", pandas_rule[0].date())

both statements put a month's bar in the hands of the reader from 2020-02-01


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

print("priced legs are read as one long table by union; the factor archives are not part of it")
print("the gate is available_from <= the moment read, and nothing else")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
priced legs are read as one long table by union; the factor archives are not part of it
the gate is available_from <= the moment read, and nothing else


In [3]:
import subprocess
import sys
from pathlib import Path

# The package is imported from the repository root, so the run needs the root as its working
# directory rather than wherever the kernel was started. Walked up from the kernel's own directory
# rather than written in at generation time: an absolute path here would name one workstation, and
# the notebook is a file every reader runs on their own.
root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "portfolio_workbench").is_dir()
)

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.data.sql"], capture_output=True, text=True, cwd=root
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[data] snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
[data] the as-of join in SQL: 2496 instrument-days visible
[data] instrument            first     last  months  visible
[data] 4GLD.DE             2010-09  2026-08     192      192
[data] EURSEK=X            2010-09  2026-08     192      192
[data] EURUSD=X            2010-09  2026-08     192      192
[data] IBCI.AS             2010-09  2026-08     192      192
[data] IBGL.AS             2010-09  2026-08     192      192
[data] IEAC.AS             2010-09  2026-08     192      192
[data] IEGE.AS             2010-09  2026-08     192      192
[data] IHYG.L              2010-09  2026-08     192      192
[data] IMEU.AS             2010-09  2026-08     192      192
[data] IWDA.AS             2010-09  2026-08     192      192
[data] IWDP.AS             2010-09  2026-08     192      192
[data] XACT-NORDEN.ST      2010-09  2026-08     192      192
[data] XEON.DE             2010-09  2026-08     192      192
[data] read at 2026-0

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

Two statements of one contract that agree are worth more than one statement, because the second can be read by someone who does not read Python and can be pointed at another project's loader. What the query does not carry is any of the quality gate: the eight stops and four warnings live in the pandas path, and a reader who took the query as the contract would satisfy the shape of the table without the checks on its content. The assembly's agreement is a tolerance rather than an identity, because the two paths compound a month's rate in a different order: on the frozen snapshot the largest month-instrument difference is zero, and the stated tolerance is what another snapshot's rerun would be read against.

## 7. What this module does not establish

Nothing here establishes that the snapshot is usable, which is the quality gate's work and not the contract's. Nothing here establishes that a second project's file layout matches: what is fixed is the columns and the availability rule, and a sibling satisfies it by writing those columns rather than by reusing these queries.